# HFD — Exercises 3.1–3.4
**Name:** Azizbek Ganiev **ID:** 475150
_University of Warsaw — Quantitative Finance_


## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.regression.quantile_regression import QuantReg
from statsmodels.tsa.stattools import adfuller, kpss, grangercausalitytests

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True

print("Env ready.")

## Data — AAPL & MSFT (CSV, local)

In [ ]:
def load_price_csv(path):
    df = pd.read_csv(path)
    # date column guess
    for c in ['Date','date','DATE','timestamp']:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c])
            df = df.set_index(c).sort_index()
            break
    else:
        raise ValueError("No date column found. Please include 'Date'.")
    # price column
    price_col = 'Adj Close' if 'Adj Close' in df.columns else ('Close' if 'Close' in df.columns else None)
    if price_col is None:
        raise ValueError("CSV needs 'Adj Close' or 'Close'.")
    return df[[price_col]].rename(columns={price_col:'price'}).dropna()

aapl = load_price_csv("AAPL.csv")
msft = load_price_csv("MSFT.csv")

# align dates
px = pd.concat({'AAPL':aapl['price'], 'MSFT':msft['price']}, axis=1).dropna()

# log returns
ret = np.log(px).diff().dropna()
ret.tail()

## Exercise 3.1 — Quantile regression (τ=0.5) + rolling β (window=250)

In [ ]:
# y: AAPL returns, x: MSFT returns
y = ret['AAPL']
x = ret['MSFT']
X = sm.add_constant(x)

# full-sample median QR
qr_full = QuantReg(y, X).fit(q=0.5)
beta_full = float(qr_full.params.get('MSFT', np.nan))
print("Full-sample β (τ=0.5):", beta_full)

# rolling QR beta
win = 250
roll_beta, roll_idx = [], []
for i in range(win, len(ret)):
    yi = y.iloc[i-win:i]
    xi = sm.add_constant(x.iloc[i-win:i])
    try:
        r = QuantReg(yi, xi).fit(q=0.5)
        roll_beta.append(float(r.params.get('MSFT', np.nan)))
    except Exception:
        roll_beta.append(np.nan)
    roll_idx.append(yi.index[-1])

roll_beta = pd.Series(roll_beta, index=pd.DatetimeIndex(roll_idx), name='beta_qr_rolling')

# plot
plt.figure()
plt.plot(roll_beta.index, roll_beta.values, label='Rolling β (window={})'.format(win))
plt.axhline(y=beta_full, linestyle='--', label='Full-sample β')
plt.title('3.1 Rolling Quantile Regression: AAPL ~ MSFT (τ=0.5)')
plt.legend()
plt.show()

qr_full.summary().tables[1]

## Exercise 3.2 — Granger causality (both directions), rolling window=250

We test AAPL → MSFT and MSFT → AAPL using lag p=2.  
For each rolling window, we record ssr_ftest p-values and count rejections at 5%.

In [ ]:
def granger_pvalue_pair(df2, cause, effect, maxlag=2):
    tmp = df2[[effect, cause]].dropna()
    try:
        res = grangercausalitytests(tmp, maxlag=maxlag, verbose=False)
        return float(res[maxlag][0]['ssr_ftest'][1])
    except Exception:
        return np.nan

win = 250
p = 2
idx = []
p_AAPL_to_MSFT = []
p_MSFT_to_AAPL = []

for i in range(win, len(ret)):
    w = ret.iloc[i-win:i]
    idx.append(w.index[-1])
    p1 = granger_pvalue_pair(w, 'AAPL', 'MSFT', maxlag=p)  # AAPL -> MSFT
    p2 = granger_pvalue_pair(w, 'MSFT', 'AAPL', maxlag=p)  # MSFT -> AAPL
    p_AAPL_to_MSFT.append(p1)
    p_MSFT_to_AAPL.append(p2)

gc = pd.DataFrame({
    'p_AAPL→MSFT': p_AAPL_to_MSFT,
    'p_MSFT→AAPL': p_MSFT_to_AAPL
}, index=pd.DatetimeIndex(idx))

# counts
alpha = 0.05
counts = (gc < alpha).sum().to_frame('rejections_5pct')
display(counts)

# plot p-values
plt.figure()
plt.plot(gc.index, gc['p_AAPL→MSFT'], label='p(AAPL→MSFT)')
plt.plot(gc.index, gc['p_MSFT→AAPL'], label='p(MSFT→AAPL)')
plt.axhline(alpha, linestyle='--', label='0.05')
plt.title('3.2 Rolling Granger p-values (p=2, window={})'.format(win))
plt.legend()
plt.show()

gc.tail()

## Exercise 3.3 — Cointegration tests (Engle–Granger with ADF on residuals) + KPSS

We test cointegration between log prices: AAPL and MSFT.  
Steps: regress log(AAPL) on log(MSFT), test residuals with ADF (want stationarity).  
Also show KPSS on residuals (stationarity null).

In [ ]:
logp = np.log(px).dropna()
y_coint = logp['AAPL']
X_coint = sm.add_constant(logp['MSFT'])

ols = sm.OLS(y_coint, X_coint).fit()
residuals = y_coint - ols.predict(X_coint)

# ADF on residuals (H0: unit root). Small p -> reject unit root -> cointegration.
adf_stat, adf_p, *_ = adfuller(residuals.dropna(), autolag='AIC')
print("Full-sample residual ADF p-value:", adf_p)

# KPSS on residuals (H0: stationarity). Small p -> reject stationarity.
kpss_stat, kpss_p, *_ = kpss(residuals.dropna(), regression='c', nlags='auto')
print("Full-sample residual KPSS p-value:", kpss_p)

# Rolling Engle–Granger (ADF on residuals per window)
win = 250
idx = []
adf_pvals = []
for i in range(win, len(logp)):
    block = logp.iloc[i-win:i]
    yi = block['AAPL']
    Xi = sm.add_constant(block['MSFT'])
    try:
        ols_i = sm.OLS(yi, Xi).fit()
        res_i = yi - ols_i.predict(Xi)
        adf_i = adfuller(res_i.dropna(), autolag='AIC')[1]
    except Exception:
        adf_i = np.nan
    adf_pvals.append(adf_i)
    idx.append(block.index[-1])

adf_roll = pd.Series(adf_pvals, index=pd.DatetimeIndex(idx), name='ADF p(residual)')

plt.figure()
plt.plot(adf_roll.index, adf_roll.values, label='ADF p(residual)')
plt.axhline(0.05, linestyle='--', label='0.05')
plt.title('3.3 Rolling Engle–Granger (ADF on residuals)')
plt.legend()
plt.show()

adf_roll.tail()

## Exercise 3.4 — Rolling Granger via SSR (manual F-test), p-values over time

We compare restricted and unrestricted OLS:
- Unrestricted: include lagged values of both series.
- Restricted: exclude lags of the "cause" variable.
Then compute F-stat and p-value.

In [ ]:
from scipy.stats import f as f_dist

def lagmat(series, p):
    return pd.concat([series.shift(i) for i in range(1, p+1)], axis=1)

def build_granger_XY(df, y_col, x_col, p):
    # y_t = a + sum b_i * y_{t-i} + sum c_i * x_{t-i} + e_t
    y = df[y_col]
    x = df[x_col]
    Ly = lagmat(y, p)
    Lx = lagmat(x, p)
    Z = pd.concat([y, Ly, Lx], axis=1).dropna()
    y_dep = Z.iloc[:, 0]
    Ly = Z.iloc[:, 1:1+p]
    Lx = Z.iloc[:, 1+p:1+2*p]
    X_ur = sm.add_constant(pd.concat([Ly, Lx], axis=1))
    X_r = sm.add_constant(Ly)  # restricted, drop Lx
    return y_dep, X_ur, X_r

def ssr(yd, X):
    m = sm.OLS(yd, X, missing='drop').fit()
    return float(np.sum(m.resid**2)), m.df_resid

def granger_F_pvalue(df, cause, effect, p):
    yd, X_ur, X_r = build_granger_XY(df, effect, cause, p)
    ur_ssr, ur_df = ssr(yd, X_ur)
    r_ssr, r_df = ssr(yd, X_r)
    q = X_ur.shape[1] - X_r.shape[1]  # number of restrictions (lags of cause)
    F = ((r_ssr - ur_ssr)/q) / (ur_ssr / ur_df)
    pval = 1 - f_dist.cdf(F, q, ur_df)
    return F, pval

p = 2
win = 250
idx = []
pvals_AAPL_to_MSFT = []
pvals_MSFT_to_AAPL = []

for i in range(win, len(ret)):
    w = ret.iloc[i-win:i]
    # returns for Granger
    try:
        F1, pv1 = granger_F_pvalue(w, cause='AAPL', effect='MSFT', p=p)  # AAPL -> MSFT
    except Exception:
        pv1 = np.nan
    try:
        F2, pv2 = granger_F_pvalue(w, cause='MSFT', effect='AAPL', p=p)  # MSFT -> AAPL
    except Exception:
        pv2 = np.nan
    idx.append(w.index[-1])
    pvals_AAPL_to_MSFT.append(pv1)
    pvals_MSFT_to_AAPL.append(pv2)

gf = pd.DataFrame({
    'p_AAPL→MSFT': pvals_AAPL_to_MSFT,
    'p_MSFT→AAPL': pvals_MSFT_to_AAPL
}, index=pd.DatetimeIndex(idx))

plt.figure()
plt.plot(gf.index, gf['p_AAPL→MSFT'], label='p(AAPL→MSFT)')
plt.plot(gf.index, gf['p_MSFT→AAPL'], label='p(MSFT→AAPL)')
plt.axhline(0.05, linestyle='--', label='0.05')
plt.title('3.4 Rolling manual Granger p-values (p=2, window=250)')
plt.legend()
plt.show()

gf.tail()